# 05 Probability-Weighted Ranking

Salvages the Bear/Base/Bull probability-weighted model from the exploratory notebook. Use this when you want expected value across assigned scenario probabilities, not scenario-by-scenario mechanics.

**Plain English:**
This notebook gives each scenario a chance of happening and calculates the average result.

**This answers the question:** "If I believe these probabilities, which choice has the best average outcome?"

Example:
If Bear has 45%, Base has 15%, and Bull has 40%, the notebook multiplies each outcome by its probability and adds them up.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tax_risk_sim import (
    build_after_tax_sensitivity,
    build_bear_recovery_cases,
    build_bear_recovery_table,
    build_probability_weighted_scenarios,
    build_stop_benchmark,
    compare_stop_reentry_vs_hold,
    sell_today_baseline,
)

pd.options.display.float_format = "{:,.2f}".format

## Inputs

Defines the probability-weighted scenario assumptions and candidate stop levels.

**Plain English:**
This is where you say how likely Bear, Base, and Bull outcomes are.

**This answers the question:** "What odds am I assigning to each future outcome?"

Example:
Bear = 45%, Base = 15%, Bull = 40% adds to 100%.

### Input Explanations

This notebook intentionally uses the older Bear/Base/Bull assumptions from the exploratory model.

**Plain English:**
These assumptions are kept separate from the stop-and-re-entry notebooks.

**This answers the question:** "Why do these inputs differ from notebook 04?"

Example:
Notebook 04 uses 35 shares; this probability notebook keeps the older 350-share example.

`scenario_cases` defines each scenario's return and probability. Probabilities must add to 1.0.

**Plain English:**
Each row says what return happens and how likely it is.

**This answers the question:** "What future outcomes am I averaging together?"

Example:
Bear return -20% with probability 45% means a -20% outcome gets 45% weight.

`stop_loss_drops` contains candidate stop-loss triggers for expected-value ranking.

**Plain English:**
These are the stop levels to compare under the scenario probabilities.

**This answers the question:** "Which stop levels should be ranked by average outcome?"

Example:
`0.10` tests a 10% stop.

In [ ]:
shares = 350
current_price = 350.0
cost_basis_per_share = 123.0
capital_gains_tax_rate = 0.26

scenario_cases = pd.DataFrame({
    "case": ["Bear", "Base", "Bull"],
    "return": [-0.20, 0.05, 0.25],
    "probability": [0.45, 0.15, 0.40],
})

stop_loss_drops = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50])

scenario_cases

## Expected Hold Value

Calculates probability-weighted after-tax value for holding through the Bear/Base/Bull outcomes.

**Plain English:**
It multiplies each scenario's result by its chance, then adds the results.

**This answers the question:** "What is holding worth on average under my probabilities?"

Example:
If Bear is worth 80,000 at 45%, Base is worth 100,000 at 15%, and Bull is worth 130,000 at 40%, the expected value is the weighted sum of those three numbers.

In [ ]:
weighted_df, weighted_summary = build_probability_weighted_scenarios(
    scenario_cases,
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)

weighted_df

### Probability-Weighted Column Explanations

`after_tax_value` is the after-tax liquidation value in that scenario.

**Plain English:**
This is the cash you keep if that scenario happens.

**This answers the question:** "How much do I have after tax in this scenario?"

Example:
If Bull happens, this column shows the after-tax value at the Bull price.

`difference_vs_selling_today` compares that scenario against selling today after tax.

**Plain English:**
This shows whether that scenario beats selling now.

**This answers the question:** "Was holding better than selling today in this scenario?"

Example:
A value of 5,000 means the scenario beats selling today by 5,000.

`weighted_after_tax_value` multiplies the scenario's after-tax value by its probability.

**Plain English:**
This is the scenario's contribution to the average.

**This answers the question:** "How much does this scenario add to expected value?"

Example:
If value is 100,000 and probability is 40%, weighted value is 40,000.

`weighted_difference_vs_selling_today` multiplies the scenario's advantage or loss by its probability.

**Plain English:**
This is the scenario's contribution to average gain or loss versus selling today.

**This answers the question:** "How much does this scenario add to the average advantage?"

Example:
If advantage is 10,000 and probability is 40%, weighted advantage is 4,000.

In [ ]:
weighted_summary

## Probability-Weighted Stop-Loss Ranking

For each stop loss, each scenario either uses the stop return if the scenario falls through the stop, or the scenario return if it does not.

**Plain English:**
This ranks stop levels by their average result under the Bear/Base/Bull probabilities.

**This answers the question:** "Which stop level has the best expected after-tax outcome?"

Example:
If the Bear scenario falls below a 10% stop, the model uses the 10% stop result for that Bear row; if Bull rises 25%, it uses the Bull result.

In [ ]:
sell_today_after_tax = weighted_summary["sell_today_after_tax_value"]
expected_hold_after_tax = weighted_summary["expected_hold_after_tax_value"]
stop_rows = []

for drop in stop_loss_drops:
    stop_return = -drop
    scenario_values = []

    for _, scenario in scenario_cases.iterrows():
        realized_return = max(scenario["return"], stop_return)
        realized_price = current_price * (1 + realized_return)
        after_tax_value = sell_today_baseline(
            shares,
            realized_price,
            cost_basis_per_share,
            capital_gains_tax_rate,
        )["sell_today_after_tax_value"]
        scenario_values.append(after_tax_value * scenario["probability"])

    expected_after_tax_value = sum(scenario_values)
    stop_price = current_price * (1 - drop)
    stop_rows.append({
        "stop_loss_drop": drop,
        "stop_price": stop_price,
        "expected_after_tax_value_with_stop": expected_after_tax_value,
        "expected_difference_vs_selling_today": expected_after_tax_value - sell_today_after_tax,
        "expected_difference_vs_no_stop_hold": expected_after_tax_value - expected_hold_after_tax,
    })

stop_expected_value_df = pd.DataFrame(stop_rows).sort_values(
    "expected_after_tax_value_with_stop",
    ascending=False,
)
stop_expected_value_df

### Stop Ranking Column Explanations

`expected_after_tax_value_with_stop` is the probability-weighted value of applying that stop rule across the Bear/Base/Bull scenarios.

**Plain English:**
This is the average cash outcome for that stop rule.

**This answers the question:** "What is this stop worth on average?"

Example:
If Bear, Base, and Bull weighted values add to 103,000, this column shows 103,000.

`expected_difference_vs_selling_today` compares the stop rule against selling today.

**Plain English:**
Positive means the stop rule averages better than selling now.

**This answers the question:** "Does this stop beat selling today on average?"

Example:
A value of 1,500 means the stop rule averages 1,500 better than selling today.

`expected_difference_vs_no_stop_hold` compares the stop rule against holding through the probability-weighted scenarios without a stop.

**Plain English:**
Positive means the stop rule averages better than simply holding.

**This answers the question:** "Does adding this stop improve the average result versus no stop?"

Example:
A value of -500 means the stop rule is 500 worse than no-stop holding on average.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    stop_expected_value_df["stop_loss_drop"] * 100,
    stop_expected_value_df["expected_difference_vs_no_stop_hold"],
    width=3,
)
ax.axhline(0, color="black", linestyle="--")
ax.set_title("Probability-Weighted Stop-Loss Advantage vs No-Stop Hold")
ax.set_xlabel("Stop-loss trigger: drop from today's price (%)")
ax.set_ylabel("Expected after-tax advantage vs no-stop hold")
ax.grid(True, axis="y", alpha=0.3)
plt.show()